# 06 — Live demo: scoring new cases and explaining them with SHAP

This notebook re-runs the three scripts behind the original live demo, one section each:

1. `scripts/score_new_cases.py`: scores 8 financial and 8 operations cases with the exported models.
2. `scripts/run_shap_explanations.py`: real SHAP explanations plus the reconciliation checks.
3. `scripts/build_driver_labels.py`: turns SHAP output into readable driver labels.

The script files in `scripts/` are kept as they were used in August 2026. Here each one is read from
disk and run with **only** its folder constants swapped for relative paths, so nothing in the
logic changes. Outputs go to `results/demo_rerun/` and are compared with the originals in `results/`.

Section 4 measures a scale mismatch found while building the web export (notebook 07). It affects
one operations feature in this 8-case demo path.

In [1]:
import warnings
warnings.filterwarnings("ignore", message="IProgress not found")  # tqdm notice from shap's import; not relevant here
import os, shutil, json
from pathlib import Path
import numpy as np
import pandas as pd

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC = Path(os.environ.get("AIT506_DIR", REPO / "data" / "raw" / "AIT506"))
RF_PKL = Path(os.environ.get("CEROP_RF_PKL", SRC / "CEROP_Live_Demo" / "models" / "cerop_operations_rf.pkl"))

RUN = REPO / "results" / "demo_rerun"
RUN.mkdir(parents=True, exist_ok=True)

# The scripts expect the 4 artifacts and the case files in fixed folders. Build that layout in a
# temporary work folder (outside the repo, so a synced folder never picks up the 1.24 GB Random
# Forest), run from there with relative paths, then copy the outputs to results/demo_rerun/.
import tempfile
WORK = Path(tempfile.gettempdir()) / "cerop_demo"
(WORK / "models").mkdir(parents=True, exist_ok=True)
(WORK / "demo_data").mkdir(parents=True, exist_ok=True)
(WORK / "train").mkdir(parents=True, exist_ok=True)
for f in ["financial_new_cases.csv", "operations_new_cases.csv"]:
    shutil.copy(REPO / "data" / "sample" / f, WORK / "demo_data" / f)
for f in ["cerop_financial_xgb.pkl", "cerop_financial_preprocessor.pkl", "cerop_operations_preprocessor.pkl"]:
    shutil.copy(REPO / "models" / f, WORK / "models" / f)
if not (WORK / "models" / "cerop_operations_rf.pkl").exists():
    try:
        os.link(RF_PKL, WORK / "models" / "cerop_operations_rf.pkl")   # hard link: no second 1.24 GB copy
    except OSError:
        shutil.copy(RF_PKL, WORK / "models" / "cerop_operations_rf.pkl")
shutil.copy(SRC / "Week 5/GROUP/Ankit/data/financial_X_train_processed.csv", WORK / "train")
shutil.copy(SRC / "Week 6/Group/data/operations_X_train_processed_v2.csv", WORK / "train")
os.chdir(WORK)

PATHS = {
    r"<AIT506_PROJECT_FOLDER>\CEROP_Live_Demo\models": "models",
    r"<AIT506_PROJECT_FOLDER>\CEROP_Live_Demo\demo_data": "demo_data",
    r"<AIT506_PROJECT_FOLDER>\Week 5\GROUP\Ankit\data\financial_X_train_processed.csv": "train/financial_X_train_processed.csv",
    r"<AIT506_PROJECT_FOLDER>\Week 6\Group\data\operations_X_train_processed_v2.csv": "train/operations_X_train_processed_v2.csv",
}

def load_script(name, extra=None):
    """Read a script from scripts/, swap folder constants (and any listed extra lines), show each swap."""
    text = (REPO / "scripts" / name).read_text(encoding="utf-8")
    swaps = dict(PATHS, **(extra or {}))
    for old, new in swaps.items():
        if old in text:
            print(f"  swap: {old!r} -> {new!r}")
            text = text.replace(old, new)
    return text

def collect(*names):
    for n in names:
        shutil.copy(WORK / "demo_data" / n, RUN / n)
    print("copied to results/demo_rerun/:", ", ".join(names))

## 1. Score the 8 + 8 demo cases

One change beyond paths. The archived script hard-codes the operations threshold as **0.39**, taken
from the project decision log, which followed the team's earlier Colab run. The Week 6 notebook in
this repo (and the exported Random Forest) selects **0.40**. The project uses 0.40 everywhere, so
that one constant is swapped here and the swap is printed.

In [2]:
src = load_script("score_new_cases.py", {"OPERATIONS_THRESHOLD = 0.39": "OPERATIONS_THRESHOLD = 0.40"})
ns = {"__name__": "demo_score"}
exec(compile(src, "score_new_cases.py", "exec"), ns)
ns["main"]()
collect("scored_output.html")

  swap: '<AIT506_PROJECT_FOLDER>\\CEROP_Live_Demo\\models' -> 'models'
  swap: '<AIT506_PROJECT_FOLDER>\\CEROP_Live_Demo\\demo_data' -> 'demo_data'
  swap: 'OPERATIONS_THRESHOLD = 0.39' -> 'OPERATIONS_THRESHOLD = 0.40'


Loaded all 4 models/preprocessors from: models



FINANCIAL LAYER — risk scores (threshold = 0.50)
case_id                  label  risk_score risk_band
FIN-006 Solara Packaging Group    0.996009      HIGH
FIN-007 Ferrovia Machine Tools    0.991318      HIGH
FIN-008   Bluepeak Electronics    0.136627     CLEAR
FIN-003   Kestrel Textiles Ltd    0.001027     CLEAR
FIN-002  Harborline Freight Co    0.000138     CLEAR
FIN-001   Meridian Alloy Works    0.000024     CLEAR
FIN-004   Pinecrest Components    0.000014     CLEAR
FIN-005  Northgate Instruments    0.000005     CLEAR

OPERATIONS LAYER — risk scores (threshold = 0.40)
case_id     label  risk_score risk_band
OPS-008 ORD-88360    0.900423      HIGH
OPS-005 ORD-88302    0.779938      HIGH
OPS-006 ORD-88318    0.467528      HIGH
OPS-007 ORD-88345    0.457766      HIGH
OPS-003 ORD-88257    0.448464      HIGH
OPS-001 ORD-88214    0.339198     WATCH
OPS-002 ORD-88231    0.325340     WATCH
OPS-004 ORD-88269    0.309357     WATCH

Wrote HTML output to: demo_data\scored_output.html
Financial 

## 2. SHAP explanations and reconciliation checks

In [3]:
src = load_script("run_shap_explanations.py")
exec(compile(src, "run_shap_explanations.py", "exec"), {"__name__": "demo_shap"})
collect("financial_shap_explanations.csv", "operations_shap_explanations.csv")

  swap: '<AIT506_PROJECT_FOLDER>\\CEROP_Live_Demo\\models' -> 'models'
  swap: '<AIT506_PROJECT_FOLDER>\\CEROP_Live_Demo\\demo_data' -> 'demo_data'
  swap: '<AIT506_PROJECT_FOLDER>\\Week 5\\GROUP\\Ankit\\data\\financial_X_train_processed.csv' -> 'train/financial_X_train_processed.csv'
  swap: '<AIT506_PROJECT_FOLDER>\\Week 6\\Group\\data\\operations_X_train_processed_v2.csv' -> 'train/operations_X_train_processed_v2.csv'


run_shap_explanations.py:17: SyntaxWarning: invalid escape sequence '\ '



FINANCIAL — computing SHAP (XGBoost, probability space)


base_value (financial): 0.032400
shap_values shape: (8, 85)
Saved: demo_data\financial_shap_explanations.csv
case_id  base_rate                          feature_1_name  feature_1_contribution                          feature_2_name  feature_2_contribution                          feature_3_name  feature_3_contribution                    feature_4_name  feature_4_contribution  final_score
FIN-001     0.0324              Total debt/Total net worth               -0.003847                             Quick Ratio               -0.002878       Retained Earnings to Total Assets               -0.002668           Profitability_Composite               -0.002299     0.000024
FIN-002     0.0324                             Quick Ratio               -0.004577              Total debt/Total net worth               -0.004177       Retained Earnings to Total Assets               -0.003341           Profitability_Composite               -0.002494     0.000138
FIN-003     0.0324              Total debt/To

base_value (operations): 0.548184
shap_values shape: (8, 27)

Grouped from 27 columns down to 22 features (one-hot dummies combined): ['numeric__Days for shipment (scheduled)', 'numeric__Benefit per order', 'numeric__Order Item Discount', 'numeric__Order Item Discount Rate', 'numeric__Order Item Profit Ratio', 'numeric__Order Item Quantity', 'numeric__Sales', 'numeric__Product Price', 'numeric__Order_Year', 'numeric__Order_Month', 'numeric__Order_DayOfWeek', 'numeric__Order_Hour', 'numeric__GSCPI', 'Type', 'Customer Country', 'Customer Segment', 'Shipping Mode', 'high_cardinality__Category Name', 'high_cardinality__Department Name', 'high_cardinality__Market', 'high_cardinality__Order Country', 'high_cardinality__Order Region']
Saved: demo_data\operations_shap_explanations.csv
case_id  base_rate                         feature_1_name  feature_1_contribution feature_2_name  feature_2_contribution      feature_3_name  feature_3_contribution                 feature_4_name  feature_4_contr

Compare the re-run SHAP tables with the ones committed in `results/` from August 2026.

In [4]:
for layer in ["financial", "operations"]:
    old = pd.read_csv(REPO / "results" / f"{layer}_shap_explanations.csv")
    new = pd.read_csv(RUN / f"{layer}_shap_explanations.csv")
    num = old.select_dtypes("number").columns
    same_names = (old.select_dtypes(exclude="number").values == new.select_dtypes(exclude="number").values).all()
    print(f"{layer}: same feature names {same_names}, max numeric diff {np.abs(old[num].values - new[num].values).max():.2e}")

financial: same feature names True, max numeric diff 0.00e+00
operations: same feature names True, max numeric diff 0.00e+00


## 3. Readable driver labels

In [5]:
src = load_script("build_driver_labels.py")
exec(compile(src, "build_driver_labels.py", "exec"), {"__name__": "demo_labels"})

  swap: '<AIT506_PROJECT_FOLDER>\\CEROP_Live_Demo\\demo_data' -> 'demo_data'

// ==== financial_shap_explanations.csv -> JS object ====
  "FIN-001": { base: 0.0324, final: 0.0000, drivers: [
    { label: "Total debt / net worth", pts: 0 },  // +0 pts
    { label: "Quick ratio", pts: 0 },  // +0 pts
    { label: "Retained earnings / assets", pts: 0 },  // +0 pts
    { label: "Profitability (composite)", pts: 0 },  // +0 pts
    { label: "81 more factors combined", pts: -2 },  // -2 pts
  ]},
  "FIN-002": { base: 0.0324, final: 0.0001, drivers: [
    { label: "Quick ratio", pts: 0 },  // +0 pts
    { label: "Total debt / net worth", pts: 0 },  // +0 pts
    { label: "Retained earnings / assets", pts: 0 },  // +0 pts
    { label: "Profitability (composite)", pts: 0 },  // +0 pts
    { label: "81 more factors combined", pts: -2 },  // -2 pts
  ]},
  "FIN-003": { base: 0.0324, final: 0.0010, drivers: [
    { label: "Total debt / net worth", pts: -1 },  // -1 pts
    { label: "Total expense 

## 4. Known issue: `Order Item Profit Ratio` reaches the model on the wrong scale

The demo path chains the Week 4 operations preprocessor into the Week 6 Random Forest and reindexes
to the model's 27 features. Notebook 07 showed that 26 of those 27 columns match the Week 6 `_v2`
training data exactly. The odd one out is `Order Item Profit Ratio`: the `_v2` data caps it at the
training IQR bounds and standard-scales the capped values, and the Week 4 preprocessor does neither.

This cell scores the 8 demo orders both ways: as the demo did it, and with that one column put on
the `_v2` scale, using the transform recovered in notebook 07 (it reproduces all 34,393 `_v2` test
values to within 1e-9). It changes no file and no model. It shows how much the mismatch moves these
8 scores.

In [6]:
import joblib
rf = joblib.load(WORK / "models" / "cerop_operations_rf.pkl")
pre = joblib.load(WORK / "models" / "cerop_operations_preprocessor.pkl")
tr = json.load(open(REPO / "results" / "web" / "operations_profit_ratio_v2_transform.json"))
cases = pd.read_csv(REPO / "data" / "sample" / "operations_new_cases.csv")
X_raw = cases.drop(columns=["case_id", "order_ref"])
X_demo = pd.DataFrame(pre.transform(X_raw), columns=pre.get_feature_names_out())[list(rf.feature_names_in_)]
X_fix = X_demo.copy()
col = "numeric__Order Item Profit Ratio"
X_fix[col] = (X_raw["Order Item Profit Ratio"].clip(tr["iqr_low"], tr["iqr_high"]) - tr["capped_train_mean"]) / tr["capped_train_std"]
p_demo = rf.predict_proba(X_demo)[:, 1]
p_fix = rf.predict_proba(X_fix)[:, 1]
cmp = pd.DataFrame({"case_id": cases["case_id"], "profit_ratio_raw": X_raw["Order Item Profit Ratio"],
                    "demo_scale": X_demo[col].round(4), "v2_scale": X_fix[col].round(4),
                    "score_as_demoed": p_demo.round(4), "score_v2_scale": p_fix.round(4),
                    "change": (p_fix - p_demo).round(4),
                    "band_change_at_0.40": (p_demo >= 0.40) != (p_fix >= 0.40)})
print(cmp.to_string(index=False))
print(f"\nlargest absolute score change: {np.abs(p_fix - p_demo).max():.4f}; orders that cross 0.40: {int(cmp['band_change_at_0.40'].sum())}")
cmp.to_csv(RUN / "operations_profit_ratio_scale_check.csv", index=False)

case_id  profit_ratio_raw  demo_scale  v2_scale  score_as_demoed  score_v2_scale  change  band_change_at_0.40
OPS-001              0.13      0.0203   -0.2439           0.3392          0.3443  0.0051                False
OPS-002             -1.50     -3.4673   -2.1158           0.3253          0.3253  0.0000                False
OPS-003              0.30      0.3840    0.4331           0.4485          0.4575  0.0091                False
OPS-004              0.11     -0.0225   -0.3236           0.3094          0.3057 -0.0036                False
OPS-005             -1.30     -3.0394   -2.1158           0.7799          0.7799  0.0000                False
OPS-006              0.10     -0.0439   -0.3634           0.4675          0.4662 -0.0013                False
OPS-007              0.09     -0.0653   -0.4032           0.4578          0.4593  0.0015                False
OPS-008              0.06     -0.1295   -0.5227           0.9004          0.8950 -0.0055                False

largest a

The site and the console do **not** use this demo path. Their operations scores and SHAP cases come
from notebook 07, which scores the `_v2` test file the model was evaluated on, so they are unaffected.
The finding is logged in `docs/engineering_log.md` and listed in `docs/limitations.md`.